<a href="https://colab.research.google.com/github/gauravprajapati9210/Machine-Learning-Projects/blob/main/Fake_and_Real_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import depandancies
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# printing stopwords in english language

print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [15]:
news_dataset = pd.read_csv("FA-KES-Dataset.csv", encoding='utf-8', encoding_errors='ignore')

In [20]:
# first five rows of the data set
news_dataset.head(5)

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [21]:
news_dataset.isnull().sum()

,0
unit_id,0
article_title,0
article_content,0
source,0
date,0
location,0
labels,0


In [23]:
#merging the source name and newz title
news_dataset["content"] = news_dataset["source"] + ' ' + news_dataset["article_title"]
print(news_dataset["content"])

0      nna Syria attack symptoms consistent with nerv...
1      nna Homs governor says U.S. attack caused deat...
2      nna Death toll from Aleppo bomb attack at leas...
3        nna Aleppo bomb blast kills six Syrian state TV
4      nna 29 Syria Rebels Dead in Fighting for Key A...
                             ...                        
799    manar Turkish Bombardment Kills 20 Civilians i...
800    manar Martyrs as Terrorists Shell Aleppos Sala...
801    manar Chemical Attack Kills Five Syrians in Al...
802    manar 5 Killed as Russian Military Chopper Sho...
803    manar Syrian Army Kills 48 ISIL Terrorists in ...
Name: content, Length: 804, dtype: object


In [26]:
#Seperating data & label
X = news_dataset.drop(columns="labels", axis = 1)
Y = news_dataset["labels"]


In [27]:
print(X,Y)

        unit_id                                      article_title  \
0    1914947530  Syria attack symptoms consistent with nerve ag...   
1    1914947532  Homs governor says U.S. attack caused deaths b...   
2    1914947533    Death toll from Aleppo bomb attack at least 112   
3    1914947534        Aleppo bomb blast kills six Syrian state TV   
4    1914947535  29 Syria Rebels Dead in Fighting for Key Alepp...   
..          ...                                                ...   
799  1965511221    Turkish Bombardment Kills 20 Civilians in Syria   
800  1965511222    Martyrs as Terrorists Shell Aleppos Salah Eddin   
801  1965511224  Chemical Attack Kills Five Syrians in Aleppo SANA   
802  1965511226  5 Killed as Russian Military Chopper Shot down...   
803  1965511231  Syrian Army Kills 48 ISIL Terrorists in Deir E...   

                                       article_content source       date  \
0    Wed 05 Apr 2017 Syria attack symptoms consiste...    nna   4/5/2017   
1    Fr

Steamming:

Stemming is the process of reducing a  word to its root word

eg. actor,actress,acting --> act

This is very important process to reduce number of words


In [28]:
port_stem = PorterStemmer()

In [31]:
def  stemming(content):
  stemmed_content = re.sub("[^a-zA-Z]"," ",content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words("english")]
  stemmed_content = " ".join(stemmed_content)
  return stemmed_content

In [32]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [33]:
print(news_dataset['content'])

0        nna syria attack symptom consist nerv agent use
1      nna hom governor say u attack cau death doesnt...
2                nna death toll aleppo bomb attack least
3         nna aleppo bomb blast kill six syrian state tv
4             nna syria rebel dead fight key aleppo road
                             ...                        
799            manar turkish bombard kill civilian syria
800      manar martyr terrorist shell aleppo salah eddin
801     manar chemic attack kill five syrian aleppo sana
802       manar kill russian militari chopper shot syria
803     manar syrian armi kill isil terrorist deir ezzor
Name: content, Length: 804, dtype: object


In [35]:
#Seperating the data and label
X = news_dataset['content'].values
Y = news_dataset['labels'].values

 Vectorizer count the number of repeating words in a documents. so that repetaiton tells that it an impotant word give a perticualr number.


In [41]:
# Converting the textual data to  numerical data

vectorizer = TfidfVectorizer() # term frequency inverse documents frequency vectorizer
vectorizer.fit(X) # feature vectors are nothing just numbers

X = vectorizer.transform(X)


In [44]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7095 stored elements and shape (804, 771)>
  Coords	Values
  (0, 21)	0.4398385037739128
  (0, 71)	0.154270763028172
  (0, 156)	0.4398385037739128
  (0, 463)	0.4143530883600743
  (0, 469)	0.31752877227058074
  (0, 671)	0.4398385037739128
  (0, 672)	0.1123974588039333
  (0, 734)	0.33268695746858934
  (1, 71)	0.12184316409612725
  (1, 95)	0.34738477945645097
  (1, 122)	0.3272563792892771
  (1, 188)	0.24114602484679362
  (1, 211)	0.34738477945645097
  (1, 299)	0.30189759411838357
  (1, 322)	0.20673629748416938
  (1, 332)	0.34738477945645097
  (1, 405)	0.30189759411838357
  (1, 469)	0.2507844619783276
  (1, 600)	0.22200068141769194
  (1, 609)	0.34738477945645097
  (2, 36)	0.21999407466746793
  (2, 71)	0.23968501507821843
  (2, 99)	0.2929163001914418
  (2, 188)	0.47437284668556384
  (2, 392)	0.3389939588680877
  :	:
  (800, 619)	0.34302181140418664
  (800, 689)	0.2062183140289721
  (801, 36)	0.24495050157124273
  (801, 71)	0.26687

In [68]:
# Spliting the data set to training and test data set .

x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size = 0.1,stratify=Y, random_state=2)


Training the Model : LogisticRegressio model

In [69]:
model = LogisticRegression()

In [70]:
model.fit(x_train,y_train)

LogisticRegression()

In [71]:
#accuracy score on the training data
x_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(x_train_prediction,y_train)

In [72]:
print("Accuracy score of the training data : ",training_data_accuracy)

Accuracy score of the training data :  0.7814661134163209


In [73]:
#accuracy score on the test data
x_test_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(x_test_prediction,y_test)

In [74]:
print("Accuracy score of the test data : ",test_data_accuracy)

Accuracy score of the test data :  0.5925925925925926


Making a Predictive system

In [79]:
x_new = x_test[14]

prediction = model.predict(x_new)
print(prediction)

if (prediction[0] == 0):
  print("The news is Real")
else:
  print("The news is Fake")

[1]
The news is Fake


In [80]:
print(y_test[14])

1
